In [ ]:
import gdown
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, decimate
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

Загрузка данных

In [ ]:
train_id = "1ckCfc-icWS_kyUDl_8f_zZTYZBqNGVGH"
test_id  = "1u06oaFZ0ReFjrSAT0t8ycV1NdcUbjiQR"

gdown.download(id=train_id, output="train.csv", quiet=False)
gdown.download(id=test_id,  output="test.csv",  quiet=False)

Downloading...
From: https://drive.google.com/uc?id=1ckCfc-icWS_kyUDl_8f_zZTYZBqNGVGH
To: /content/train.csv
100%|██████████| 28.4M/28.4M [00:00<00:00, 240MB/s]
Downloading...
From: https://drive.google.com/uc?id=1u06oaFZ0ReFjrSAT0t8ycV1NdcUbjiQR
To: /content/test.csv
100%|██████████| 2.13M/2.13M [00:00<00:00, 112MB/s]


'test.csv'

Проверка формата

In [ ]:
train_path = "train.csv"
test_path  = "test.csv"

def load_csv_with_checks(path, has_class: bool):
    df = pd.read_csv(path, sep=";", engine="python")

    empty_rows = df.isna().all(axis=1).sum()
    if empty_rows > 0:
        print(f"{path}: найдено пустых строк: {empty_rows}.")

    nan_cols = df.isna().sum()
    bad = nan_cols[nan_cols > 0]
    if len(bad) > 0:
        print(f"{path}: есть NaN в столбцах:")
        print(bad.sort_values(ascending=False).head(100))

    if has_class:
        y = df["class"].astype(int).to_numpy()
        X_df = df.drop(columns=["class"])
    else:
        y = None
        X_df = df

    X_num = X_df.apply(pd.to_numeric, errors="coerce")

    X = X_num.to_numpy(dtype=np.float32)
    return X, y, df

X_train_raw, y, train_df = load_csv_with_checks(train_path, has_class=True)
X_test_raw, _, test_df   = load_csv_with_checks(test_path,  has_class=False)

print("Train:", X_train_raw.shape, "y:", y.shape)
print("Test:",  X_test_raw.shape)
print("Classes balance:", np.bincount(y))

Train: (1000, 1500) y: (1000,)
Test: (75, 1500)
Classes balance: [661 339]


Reshape и препроцессинг

In [ ]:
n_channels = 6
sfreq = 250
n_times = 250

X_3d = X_train_raw.reshape(-1, n_channels, n_times)
X_test_3d = X_test_raw.reshape(-1, n_channels, n_times)

def baseline_correct(X_3d, sfreq=250, baseline_ms=100):
    baseline_samples = int(sfreq * (baseline_ms / 1000.0))
    base = X_3d[:, :, :baseline_samples].mean(axis=2, keepdims=True)
    return X_3d - base

X_p = baseline_correct(X_3d, sfreq=sfreq, baseline_ms=100)
X_test_p = baseline_correct(X_test_3d, sfreq=sfreq, baseline_ms=100)

X2 = X_p.reshape(X_p.shape[0], -1)
X2_test = X_test_p.reshape(X_test_p.shape[0], -1)

print(X2.shape, X2_test.shape)

(1000, 1500) (75, 1500)


Эксперименты

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    "LogReg": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=3000, class_weight="balanced"))
    ]),
    "KNN(k=7)": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", KNeighborsClassifier(n_neighbors=7))
    ]),
    "RandomForest": RandomForestClassifier(
        n_estimators=400, random_state=42, n_jobs=-1
    ),
    "GradBoost": GradientBoostingClassifier(random_state=42),
}

results = []
for name, model in models.items():
    cv_out = cross_validate(
        model, X2, y, cv=cv,
        scoring={"acc": "accuracy", "auc": "roc_auc"},
        return_train_score=False
    )
    results.append({
        "model": name,
        "acc_mean": cv_out["test_acc"].mean(),
        "acc_std":  cv_out["test_acc"].std(),
        "auc_mean": cv_out["test_auc"].mean(),
        "auc_std":  cv_out["test_auc"].std(),
    })

res_df = pd.DataFrame(results).sort_values(by=["auc_mean", "acc_mean"], ascending=False)
res_df

,model,acc_mean,acc_std,auc_mean,auc_std
0,LogReg,0.735,0.012649,0.793722,0.013823
3,GradBoost,0.717,0.010770,0.716953,0.033359
2,RandomForest,0.677,0.009274,0.656211,0.031923
1,KNN(k=7),0.640,0.027019,0.551214,0.027169


Выбор лучшего классификатора

In [ ]:
best_name = res_df.iloc[0]["model"]
best_model = models[best_name]
print("Best:", best_name)

best_model.fit(X2, y)
pred = best_model.predict(X2_test).astype(int)

Best: LogReg


Сохранение ответа

In [ ]:
with open("answer_simple.txt", "w", encoding="utf-8") as f:
    f.write("".join(map(str, pred.tolist())))